# 3c – LISA su dati puntuali con pesi kNN (Visium HNE)

**SPEC 2026** – Econometria Spaziale (Python Lab)

Equivalente Python di `R/3_lisa_visium_hne.R`

- Pesi kNN (k = 8) su dati puntuali
- Moran’s I globale su `log(1 + Resp18)`
- LISA e mappa cluster

In [ ]:
!pip install -q geopandas pyarrow libpysal esda mapclassify

In [ ]:
import geopandas as gpd
import numpy as np
import os
import matplotlib.pyplot as plt
from shapely.geometry import LineString
from libpysal.weights import KNN, lag_spatial
from esda import Moran, Moran_Local
import warnings
warnings.filterwarnings('ignore')
np.random.seed(123)

BASE = "https://github.com/vincnardelli/spec/raw/main/data"
for f in ["tanzania.parquet", "kc_house.parquet", "kc_grid.parquet",
          "italian_provinces.parquet", "visium_hne_points.parquet"]:
    if not os.path.exists(f):
        !wget -q {BASE}/{f}

## 1) Caricamento dati

In [ ]:
hne = gpd.read_parquet("visium_hne_points.parquet")
hne["logResp18"] = np.log1p(hne["Resp18"])
y = hne["logResp18"].values
hne.head()

## 2) Visualizzazione esplorativa

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
hne.plot(column="Resp18", legend=True, cmap="viridis",
         markersize=5, alpha=0.9, ax=ax)
ax.set_title("Visium HNE \u2013 Resp18")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 3) Pesi spaziali \u2013 kNN (k = 8)

In [ ]:
coords = np.column_stack([hne.geometry.x, hne.geometry.y])
w = KNN.from_array(coords, k=8)
w.transform = "r"
print(w)

In [ ]:
lines = []
xs, ys = hne.geometry.x.values, hne.geometry.y.values
for i in w.neighbors:
    for j in w.neighbors[i]:
        lines.append(LineString([(xs[i], ys[i]), (xs[j], ys[j])]))
network = gpd.GeoDataFrame(geometry=lines, crs=hne.crs)

fig, ax = plt.subplots(figsize=(8, 8))
network.plot(ax=ax, color="grey", linewidth=0.2, alpha=0.5)
hne.plot(ax=ax, color="black", markersize=1, alpha=0.7)
ax.set_title("Rete kNN (k=8)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4) Moran\u2019s I globale

In [ ]:
mi = Moran(y, w, permutations=999)
print(f"Moran's I:       {mi.I:.4f}")
print(f"Expected I:      {mi.EI:.4f}")
print(f"p-value (norm):  {mi.p_norm:.4f}")
print(f"p-value (sim):   {mi.p_sim:.4f}")

## 5) LISA (Local Moran\u2019s I)

In [ ]:
lisa = Moran_Local(y, w, permutations=999)

q_labels = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
hne["lisa_cluster"] = [
    q_labels[q] if p < 0.05 else "Not significant"
    for q, p in zip(lisa.q, lisa.p_sim)
]

colors = {
    "High-High": "#B2182B", "Low-Low": "#2166AC",
    "High-Low": "#EF8A62",  "Low-High": "#67A9CF",
    "Not significant": "#bfbfbf"
}

fig, ax = plt.subplots(figsize=(8, 8))
hne.plot(color=hne["lisa_cluster"].map(colors),
         markersize=5, alpha=0.9, ax=ax)
handles = [plt.Rectangle((0,0),1,1, facecolor=c) for c in colors.values()]
ax.legend(handles, colors.keys(), title="LISA cluster", loc="lower left")
ax.set_title("LISA Cluster Map \u2013 log(1+Resp18)")
ax.set_axis_off()
plt.tight_layout()
plt.show()